# 41. 정상 범위 중심 준지도(RG3) + 약한 지도 M4 전체 기간 재시도

**원칙**: unlabeled는 불량을 끌어오지 않고 **정상 범위를 배우는 데만** 사용 (의사 불량 라벨·라벨 전파 제외)

### A. RG3 — 가스 RF와 같은 변수(std 24개)로 공정 비교
| 방법 | unlabeled 사용 방식 |
|---|---|
| M0 supervised | 안 씀 (기준 = 37번 가스 RF 설정) |
| N1 normal_augment | unlabeled 3,000샷을 정상으로 학습에 추가 (가중치 0.3) |
| N2 normal_distance_feat | unlabeled로만 학습한 '정상에서 떨어진 정도' 2개(Mahalanobis·IsolationForest)를 변수로 추가 |
| N3 weighted_deviation | 정상 기준 robust z의 제곱(변수별 정상 이탈 크기)으로 로지스틱 회귀 → 어떤 변수의 이탈이 중요한지 해석 |
| N4 pretrain_dae | unlabeled로 오토인코더 학습 → 정상 표현 + 복원오차를 변수로 추가 |
| N5 self_train_negatives | '확실한 정상'만 의사 라벨로 추가 (불량 의사 라벨 없음) |

### B. M4 약한 지도 — unlabeled 전체 기간
- 40번은 날짜가 6일·4일뿐이라 과적합 → 전체 기간 생산 샷으로 날짜를 늘림
- **labeled 날짜는 학습에서 제외**(평가 독립), 변수는 미리 정한 1~2개(최종 모델 점수 / std4 흔들림)
- **날짜 빼고 예측 ρ가 유의해야** labeled 평가를 의미 있게 봄. 학습 없는 단순 상관도 같이 표시

**신뢰성**: A는 Run 하나씩 빼고 검증 + 같은 fold의 M0 대비 Δ + 시드 3개. B는 labeled 라벨을 학습에 전혀 안 씀
**실행 시간**: A 약 10~15분(시드 3개), B 수 분

In [1]:
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
import semi_normal as N

## A. RG3 정상 범위 중심 준지도

In [2]:
A = N.run_A('rg3', seeds=(0, 1, 2))
display(A['summary'].round(3))

[rg3] 변수 24개 (std10만) | unlabeled 4,359샷
  seed 0 완료
  seed 1 완료
  seed 2 완료


,타깃,방법,② AP,AP_가스,AP_미성형,가스 평균,가스 범위,미성형 평균,미성형 범위,Δ② AP 평균,Δ② AP>0 시드,Δ가스 평균,Δ미성형 평균
0,가스,M0_supervised,0.119,0.110,0.035,7.667,7~8,1.667,1~2,0.000,0,0.000,0.000
1,가스,N1_normal_augment,0.086,0.079,0.024,5.000,5~5,0.333,0~1,-0.032,0,-2.667,-1.333
2,가스,N2_normal_distance_feat,0.100,0.091,0.028,6.333,6~7,1.333,1~2,-0.019,0,-1.333,-0.333
3,가스,N3_weighted_deviation,0.076,0.073,0.019,3.000,3~3,0.000,0~0,-0.042,0,-4.667,-1.667
4,가스,N4_pretrain_dae,0.094,0.083,0.028,5.333,4~7,1.667,1~2,-0.024,0,-2.333,0.000
5,가스,N5_self_train_negatives,0.091,0.090,0.022,3.000,3~3,1.000,1~1,-0.028,0,-4.667,-0.667
6,미성형,M0_supervised,0.077,0.058,0.032,3.333,3~4,1.333,1~2,0.000,0,0.000,0.000
7,미성형,N1_normal_augment,0.102,0.058,0.092,4.000,4~4,3.000,3~3,0.025,3,0.667,1.667
8,미성형,N2_normal_distance_feat,0.066,0.049,0.028,2.667,2~4,1.333,1~2,-0.011,0,-0.667,0.000
9,미성형,N3_weighted_deviation,0.057,0.042,0.020,0.000,0~0,0.000,0~0,-0.020,0,-3.333,-1.333


### N3 가중치 — 어떤 변수의 '정상 이탈'이 가스·미성형과 관련 있나 (전체 labeled로 학습, 해석용)

In [3]:
C = A['n3_coefs']
for t in ('가스', '미성형'):
    cols = [c for c in C.columns if c.startswith(t)]
    print(f'===== {t} (시드 평균 가중치 상위 8) =====')
    display(C[cols].mean(1).sort_values(ascending=False).head(8).round(3))

===== 가스 (시드 평균 가중치 상위 8) =====


C:\Users\한국전파진흥협회\AppData\Local\Temp\ipykernel_7408\3409145143.py:5: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  display(C[cols].mean(1).sort_values(ascending=False).head(8).round(3))


Plasticizing_Time|std10       0.691
Cushion_Position|std10        0.473
Filling_Time|std10            0.461
Barrel_Temperature_4|std10    0.345
Clamp_Close_Time|std10        0.238
Barrel_Temperature_6|std10    0.149
Hopper_Temperature|std10      0.143
Barrel_Temperature_1|std10    0.118
dtype: float64

===== 미성형 (시드 평균 가중치 상위 8) =====


C:\Users\한국전파진흥협회\AppData\Local\Temp\ipykernel_7408\3409145143.py:5: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  display(C[cols].mean(1).sort_values(ascending=False).head(8).round(3))


Filling_Time|std10              1.135
Max_Injection_Speed|std10       0.705
Barrel_Temperature_5|std10      0.541
Average_Back_Pressure|std10     0.541
Barrel_Temperature_6|std10      0.375
Barrel_Temperature_1|std10      0.284
Max_Injection_Pressure|std10    0.255
Plasticizing_Time|std10         0.217
dtype: float64

In [4]:
display(A['by_seed'].round(3))

,제품,방법,타깃,seed,② AP,AP_가스,AP_미성형,상위10% 가스,상위10% 미성형,Run별10% 가스,Run별10% 미성형,학습 못 한 Run 수,Δ② AP,ΔAP_가스,ΔAP_미성형,Δ상위10% 가스,Δ상위10% 미성형
0,rg3,M0_supervised,가스,0,0.110,0.096,0.036,8,2,7,1,0,0.000,0.000,0.000,0,0
1,rg3,N1_normal_augment,가스,0,0.081,0.070,0.024,5,1,4,1,0,-0.029,-0.026,-0.012,-3,-1
2,rg3,N2_normal_distance_feat,가스,0,0.100,0.094,0.026,7,1,7,1,0,-0.010,-0.002,-0.010,-1,-1
3,rg3,N3_weighted_deviation,가스,0,0.076,0.073,0.019,3,0,3,1,0,-0.034,-0.023,-0.017,-5,-2
4,rg3,N4_pretrain_dae,가스,0,0.105,0.095,0.030,7,1,7,2,0,-0.005,-0.001,-0.005,-1,-1
5,rg3,N5_self_train_negatives,가스,0,0.077,0.071,0.022,3,1,2,1,0,-0.033,-0.025,-0.014,-5,-1
6,rg3,M0_supervised,미성형,0,0.074,0.059,0.027,3,1,4,2,0,0.000,0.000,0.000,0,0
7,rg3,N1_normal_augment,미성형,0,0.087,0.056,0.049,4,3,5,3,0,0.013,-0.002,0.022,1,2
8,rg3,N2_normal_distance_feat,미성형,0,0.068,0.050,0.030,4,1,3,2,0,-0.006,-0.009,0.002,1,0
9,rg3,N3_weighted_deviation,미성형,0,0.057,0.042,0.020,0,0,4,0,0,-0.017,-0.017,-0.007,-3,-1


## B. M4 약한 지도 — 전체 기간 (CN7·RG3)

In [5]:
B = N.run_B(('cn7', 'rg3'))
print('날짜 수준 검증 (ρ·p가 유의해야 의미 있음)')
display(B['daily'].round(3))
print('labeled 평가 (labeled 라벨은 학습에 미사용)')
display(B['labeled_eval'].round(3))

[cn7] 날짜 15일 (labeled 날짜 3일 제외 → 학습 12일)
  [cn7] 완료
[rg3] 날짜 21일 (labeled 날짜 3일 제외 → 학습 18일)
  [rg3] 완료
날짜 수준 검증 (ρ·p가 유의해야 의미 있음)


,제품,모형,학습 날짜,가중치,날짜 빼고 예측 ρ,p,단순 상관 ρ(final),p(final),단순 상관 ρ(std4),p(std4)
0,cn7,w1_final,12,[0.137],-0.825,0.001,-0.280,0.379,0.007,0.983
1,cn7,w2_std4,12,[-0.125],-0.615,0.033,-0.280,0.379,0.007,0.983
2,cn7,w12_both,12,"[0.144, -0.16]",-0.930,0.000,-0.280,0.379,0.007,0.983
3,rg3,w1_final,18,[-2.822],0.083,0.745,-0.165,0.512,0.496,0.036
4,rg3,w2_std4,18,[0.068],-0.433,0.073,-0.165,0.512,0.496,0.036
5,rg3,w12_both,18,"[-7.06, 3.083]",0.347,0.158,-0.165,0.512,0.496,0.036


labeled 평가 (labeled 라벨은 학습에 미사용)


,제품,방법,타깃,seed,② AP,AP_가스,AP_미성형,상위10% 가스,상위10% 미성형,Run별10% 가스,Run별10% 미성형,학습 못 한 Run 수
0,cn7,M4_full|w1_final,불량전체,0,0.162,0.007,0.563,2,3,1,3,0
1,cn7,M4_full|w2_std4,불량전체,0,0.006,0.004,0.003,0,0,1,2,0
2,cn7,M4_full|w12_both,불량전체,0,0.208,0.005,0.751,1,3,0,3,0
3,rg3,M4_full|w1_final,불량전체,0,0.038,0.028,0.014,1,1,0,1,0
4,rg3,M4_full|w2_std4,불량전체,0,0.060,0.046,0.027,1,1,1,1,0
5,rg3,M4_full|w12_both,불량전체,0,0.061,0.031,0.075,1,1,0,1,0


In [6]:
for p, d in B['days'].items():
    print(p, '날짜별 샷 수·불량 수'); display(d.assign(샷당불량률=d['err'] / (2 * d['n'])).round(4))

cn7 날짜별 샷 수·불량 수


,n,err,labeled_day,샷당불량률
plan_date,,,,
2020-04-04,1198,17.0,False,0.0071
2020-05-27,590,2.0,False,0.0017
2020-07-29,478,15.0,False,0.0157
2020-07-30,107,7.0,False,0.0327
2020-08-26,173,4.0,False,0.0116
2020-08-28,125,0.0,False,0.0000
2020-09-07,712,6.0,False,0.0042
2020-09-19,398,10.0,False,0.0126
2020-09-21,214,155.0,False,0.3621


rg3 날짜별 샷 수·불량 수


,n,err,labeled_day,샷당불량률
plan_date,,,,
2020-03-29,582,485.0,False,0.4167
2020-03-30,1447,126.0,False,0.0435
2020-04-24,582,3.0,False,0.0026
2020-04-25,103,0.0,False,0.0000
2020-04-27,664,8.0,False,0.0060
2020-04-28,602,3.0,False,0.0025
2020-04-29,512,7.0,False,0.0068
2020-06-19,424,16.0,False,0.0189
2020-06-23,504,67.0,False,0.0665
